# Loan payback prediction machine 

In [ ]:
# imports 
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Pull the data and inspect it 
train_data = pd.read_csv('train.csv')
test_data = pd.read_csv('test.csv')

In [12]:
# Check the dtypes of the test data 
test_data.dtypes

id                        int64
annual_income           float64
debt_to_income_ratio    float64
credit_score              int64
loan_amount             float64
interest_rate           float64
gender                   object
marital_status           object
education_level          object
employment_status        object
loan_purpose             object
grade_subgrade           object
dtype: object

In [2]:
# Pull the object columns from the train data 
train_data.select_dtypes(include=['object']).columns.tolist()

['gender',
 'marital_status',
 'education_level',
 'employment_status',
 'loan_purpose',
 'grade_subgrade']

In [3]:
# Pull the object columns from the test data 
test_data.select_dtypes(include=['object']).columns.tolist()

['gender',
 'marital_status',
 'education_level',
 'employment_status',
 'loan_purpose',
 'grade_subgrade']

In [4]:
# create a cardinality function to check the number of unique values in a column for the object dtypes. 
def cardinality(df, col): 
    return df[col].nunique()

# Check the cardinality of the object columns in the train data 
for col in train_data.select_dtypes(include=['object']).columns:
    print(f' {col}: {cardinality(train_data, col)}')

 gender: 3
 marital_status: 4
 education_level: 5
 employment_status: 5
 loan_purpose: 8
 grade_subgrade: 30


In [5]:
# Check the cardinality of the object columns in the test data
for col in test_data.select_dtypes(include=['object']).columns:
    print(f' {col}: {cardinality(test_data, col)}')
    

 gender: 3
 marital_status: 4
 education_level: 5
 employment_status: 5
 loan_purpose: 8
 grade_subgrade: 30


In [ ]:
def one_hot_encode(train_df, test_df, cat_cols, drop_original=True):
    """
    Fit one-hot encoding on train only, then transform train and test.
    Returns encoded train/test DataFrames with numeric columns only.
    """
    from sklearn.preprocessing import OneHotEncoder

    encoder = OneHotEncoder(
        sparse_output=False,
        handle_unknown="ignore",  # unseen test categories -> all zeros for that column
    )
    encoder.fit(train_df[cat_cols])

    train_encoded = encoder.transform(train_df[cat_cols])
    test_encoded = encoder.transform(test_df[cat_cols])

    feature_names = encoder.get_feature_names_out(cat_cols)
    train_ohe = pd.DataFrame(train_encoded, columns=feature_names, index=train_df.index)
    test_ohe = pd.DataFrame(test_encoded, columns=feature_names, index=test_df.index)

    num_cols = train_df.select_dtypes(include=["number"]).columns.tolist()
    if "loan_paid_back" in num_cols:
        num_cols.remove("loan_paid_back")  # keep target separate

    train_out = pd.concat([train_df[num_cols], train_ohe], axis=1)
    test_out = pd.concat([test_df[num_cols], test_ohe], axis=1)

    if drop_original:
        train_out = train_out.drop(columns=cat_cols, errors="ignore")
        test_out = test_out.drop(columns=cat_cols, errors="ignore")

    return train_out, test_out, encoder


cat_cols = [
    "gender",
    "marital_status",
    "education_level",
    "employment_status",
    "loan_purpose",
    "grade_subgrade",  # 30 levels — include if you want; or drop for fewer features
]

train_encoded, test_encoded, ohe = one_hot_encode(train_data, test_data, cat_cols)

print(train_encoded.shape, test_encoded.shape)
train_encoded.head()_hot_encode in features: 
    train_data = 


TypeError: OneHotEncoder.__init__() got an unexpected keyword argument 'sparse'